# Multicollinearity, OLS Instability, and Regularization

## Objective

In this lab, you will **observe how multicollinearity destabilizes ordinary least squares (OLS)** and how **pseudo-inverse** and **ridge regression** improve numerical stability. All results are based on a small synthetic dataset with nearly identical features.

---

## Data Setup (Given)

You are given a dataset with:

* (n = 100) samples
* (p = 3) features

The features are constructed to be **almost perfectly correlated**:
$[
x_2 \approx x_1, \quad x_3 \approx x_1
]$

The response variable is generated using known true weights:
$[
\mathbf{w}_{\text{true}} = [1.0,,-1.0,,0.5]
]$

Small Gaussian noise is added to the output.

---

## Part 1: OLS and Ill-Conditioning

OLS uses the **normal equation**:
$[
\hat{\mathbf{w}}_{\text{OLS}} = (\mathbf{X}^\top \mathbf{X})^{-1}\mathbf{X}^\top \mathbf{y}
]$

### What You Compute

* $(\mathbf{X}^\top \mathbf{X})$
* Its inverse
* The **condition number** of $(\mathbf{X}^\top \mathbf{X})$

### Key Observation

* The condition number is extremely large
* This indicates that (\mathbf{X}^\top \mathbf{X}) is **ill-conditioned**
* As a result, OLS coefficients are **numerically unstable**

Even though the data is generated from known true weights, the OLS estimates differ substantially.

---

## Part 2: Sensitivity to Tiny Perturbations

You add a **very small perturbation** to the response:
$[
\mathbf{y}_{\text{perturbed}} = \mathbf{y} + 10^{-6}\boldsymbol{\epsilon}
]$

You then recompute OLS coefficients.

### What You Measure

*$ ( |\Delta \mathbf{y}| )$
* $( |\Delta \mathbf{w}| )$
* **Amplification factor**:
 $ [
  \frac{|\Delta \mathbf{w}|}{|\Delta \mathbf{y}|}
  ]$

### Key Observation

* A tiny change in $(\mathbf{y})$ produces a **huge change in $(\mathbf{w})$**
* This confirms that OLS is **highly sensitive** under multicollinearity

---

## Part 3: Pseudo-Inverse as a Remedy

You replace the normal equation with the **Moore–Penrose pseudo-inverse**:
$[
\hat{\mathbf{w}}_{\text{pinv}} = \mathbf{X}^{+}\mathbf{y}
]$

### What Changes

* The pseudo-inverse avoids directly inverting $(\mathbf{X}^\top \mathbf{X})$
* It relies on **singular value decomposition (SVD)**

### Key Observation

* Coefficients are much more stable
* Perturbations in $(\mathbf{y})$ cause **far smaller changes** in $(\mathbf{w})$
* The amplification factor drops dramatically

---

## Part 4: Ridge Regression

Ridge regression modifies the normal equation:
$[
\hat{\mathbf{w}}_{\text{ridge}} =
(\mathbf{X}^\top \mathbf{X} + \lambda \mathbf{I})^{-1}\mathbf{X}^\top \mathbf{y}
]$

You compute ridge solutions for:

* Small $(\lambda)$
* Large $(\lambda)$

### Key Observations

* Adding $(\lambda \mathbf{I})$ stabilizes the inversion
* Larger $(\lambda)$ shrinks coefficients more aggressively
* Ridge coefficients are **more stable but biased**

---

## Summary of What You Learned

* Nearly identical features create **severe multicollinearity**
* Multicollinearity makes OLS **numerically unstable**
* Small output noise can cause **large coefficient changes**
* The pseudo-inverse improves stability without explicit regularization
* Ridge regression stabilizes solutions by trading bias for variance reduction

---

## Reflection Questions

1. Why does a large condition number indicate numerical instability?
2. How does increasing $(\lambda)$ change ridge coefficients?
3. When would ridge regression be preferable to OLS in practice?


In [41]:
import numpy as np
# Generate highly-correlated data (x1, x2, x3): 100x3
np.random.seed(42)
n = 100        # number of samples
p = 3          # number of features
x1 = np.random.randn(n)
x2 = x1 + 1e-6 * np.random.randn(n)
x3 = x1 - 1e-6 * np.random.randn(n)
# Design Matrix
X = np.column_stack([x1, x2, x3])
# True coefs/weigths
true_w = np.array([1.0, -1.0, 0.5])
# Small noise
noise = 0.01 * np.random.randn(n)
# Generaete y
y = X @ true_w + noise

# Inverting XtX (X-transposed X) should give ill conditioned matrix
XtX = X.T @ X
print("XtX_Inv\n", np.linalg.inv(XtX))
print("condition number\n", np.linalg.cond(XtX))
# Go ahead and apply normal equation
w_ols = np.linalg.inv(X.T @ X) @ X.T @ y
print("w_ols\n", w_ols)
# true weights ?
print("true_w\n", true_w)

# Add little noise to y
y_perturbed = y + 1e-6 * np.random.randn(n)
# Apply normal equation again
w_ols_perturbed = np.linalg.inv(X.T @ X) @ X.T @ y_perturbed
# verify that adding little noise has drastic effect on solution
print("w_ols\n", w_ols)
print("w_ols_perturbed\n", w_ols_perturbed)
# measure delta to quantify the noise effect
delta_y = y_perturbed - y
delta_w = w_ols_perturbed - w_ols
print("||Δy||:", np.linalg.norm(delta_y))
print("||Δw||:", np.linalg.norm(delta_w))
print("Amplification factor:", np.linalg.norm(delta_w) / np.linalg.norm(delta_y))
# Solution ?

# 1. apply pseudo-inverse
w_pinv = np.linalg.pinv(X) @ y
w_pinv_perturbed = np.linalg.pinv(X) @ y_perturbed
print("\nAfter pseudo-inv")
print("w_pinv\n", w_pinv)
print("w_pinv_perturbed\n", w_pinv_perturbed)
# Check the delta now
delta_w_pinv = w_pinv_perturbed - w_pinv
print("||Δy||:", np.linalg.norm(delta_y))
print("||Δw||:", np.linalg.norm(delta_w_pinv))
print("Amplification factor:", np.linalg.norm(delta_w_pinv) / np.linalg.norm(delta_y))
# apply ridge regresion
lambda_ = 0.01
I = np.eye(p)
w_ridge = np.linalg.inv(X.T @ X + lambda_ * I) @ X.T @ y
# stronger lambda
lambda_ = 10
w_ridge_stronger = np.linalg.inv(X.T @ X + lambda_ * I) @ X.T @ y
# compare
print("True w:      ", true_w)
print("OLS:         ", w_ols)
print("Pseudo-inv:  ", w_pinv)
print("Ridge λ=0.1: ", w_ridge)
print("Ridge λ=1.0: ", w_ridge_stronger)

XtX_Inv
 [[ 1.99867766e+10 -1.12214779e+10 -8.76530224e+09]
 [-1.12214779e+10  1.13198540e+10 -9.83744284e+07]
 [-8.76530224e+09 -9.83744284e+07  8.86367857e+09]]
condition number
 7489949577251.679
w_ols
 [ 712.86701842 -375.62848445 -336.74035996]
true_w
 [ 1.  -1.   0.5]
w_ols
 [ 712.86701842 -375.62848445 -336.74035996]
w_ols_perturbed
 [ 712.58783496 -375.43428753 -336.65537353]
||Δy||: 1.059878991427426e-05
||Δw||: 0.35054035750860246
Amplification factor: 33073.62070046327

After pseudo-inv
w_pinv
 [ 712.22385991 -375.39910922 -336.32664734]
w_pinv_perturbed
 [ 711.94489501 -375.20499207 -336.2417997 ]
||Δy||: 1.059878991427426e-05
||Δw||: 0.35028845663020813
Amplification factor: 33049.853753440846
True w:       [ 1.  -1.   0.5]
OLS:          [ 712.86701842 -375.62848445 -336.74035996]
Pseudo-inv:   [ 712.22385991 -375.39910922 -336.32664734]
Ridge λ=0.1:  [0.16607254 0.16606916 0.16606866]
Ridge λ=1.0:  [0.15964453 0.1596445  0.15964449]


## Orthogonality of residuals and $X^Tw$

In [45]:
residual = y - X @ w_ols
print(np.linalg.norm(X.T @ residual))

residual = y - X @ w_pinv
print(np.linalg.norm(X.T @ residual))

residual = y - X @ w_ridge
print(np.linalg.norm(X.T @ residual))


0.010142788035917921
3.2636474524057394e-09
0.0028764187987537415


## Notes
- Condition number: a measure of how sensitive the solution of a linear system is to small changes in the input. More specifically, for a given matrix, the condition number is the ratio of the largest singular value to the smallest singular value.
- Ill-conditioning does not mean “always unstable”
- It means “arbitrarily sensitive to small perturbations”